# Integrator regression vs JPL Horizons

Pass a real asteroid's osculating elements (from Horizons) into the production n-body
integrator `propagate_elements_nbody` (`src/velocity_density_pipeline_gmm.py`), propagate,
and compare the result to Horizons at the target time — both the **state** (position) and
the recovered **orbital elements**.

Example object: **2024 YR4**. Change `OBJECT` / `EPOCH_MJD` / `TARGET_MJD` to test others.

**Runs on Hyak** (needs `assist` + `rebound` + `sorcha` + the ASSIST kernel). The Horizons
cells run anywhere `astroquery` has network; the propagation cell is the only Hyak-only one.

In [1]:
import sys, numpy as np
sys.path.insert(0, "../../src")   # for velocity_density_pipeline_gmm
from astroquery.jplhorizons import Horizons
from astropy.time import Time

OBJECT     = "2024 YR4"
EPOCH_MJD  = 60600.0     # TDB: epoch at which we take the elements (start of propagation)
TARGET_MJD = 61642.0     # TDB: propagate to here, then compare to Horizons
GR_MODEL   = "GR_SIMPLE" # 'GR_SIMPLE' = Sorcha parity; 'GR_EIH' = ASSIST full model

AU_KM   = 149597870.7
DAY_S   = 86400.0
GM_SUN  = 1.32712440018e11   # km^3/s^2 (for the state->elements check only)
print(f"{OBJECT}: propagate elements @ MJD {EPOCH_MJD} (TDB) -> MJD {TARGET_MJD} (TDB), span {(TARGET_MJD-EPOCH_MJD)/365.25:.2f} yr")

2024 YR4: propagate elements @ MJD 60600.0 (TDB) -> MJD 61642.0 (TDB), span 2.85 yr


## 1. Elements from Horizons at the start epoch

In [2]:
def horizons_elements(obj, mjd_tdb):
    """Heliocentric ecliptic J2000 osculating elements at mjd_tdb."""
    q = Horizons(id=obj, location='@sun', epochs=2400000.5 + mjd_tdb, id_type='smallbody')
    el = q.elements(refsystem='J2000', refplane='ecliptic')[0]
    return dict(a=float(el['a']), e=float(el['e']), incl=float(el['incl']),
                Omega=float(el['Omega']), w=float(el['w']),
                Tp_mjd=float(el['Tp_jd']) - 2400000.5, epoch_mjd=float(el['datetime_jd']) - 2400000.5)

el0 = horizons_elements(OBJECT, EPOCH_MJD)
for k, v in el0.items():
    print(f"  {k:9s} = {v}")

  a         = 2.539084380929908
  e         = 0.6641390888394662
  incl      = 3.452781847706646
  Omega     = 271.412339198305
  w         = 134.6422411098297
  Tp_mjd    = 60636.6354464991
  epoch_mjd = 60600.0


## 2. Propagate through the integrator  (Hyak-only)

In [3]:
import velocity_density_pipeline_gmm as vdp

r_km, v_kms = vdp.propagate_elements_nbody(
    a_AU=[el0['a']], e=[el0['e']], inc_deg=[el0['incl']],
    raan_deg=[el0['Omega']], argp_deg=[el0['w']],
    tp_mjd=[el0['Tp_mjd']], epoch_mjd=[el0['epoch_mjd']],
    obstime_str=Time(TARGET_MJD, format='mjd', scale='tdb'),
    show_progress=False, gr_model=GR_MODEL,
)
r_int = r_km[0]                    # heliocentric ecliptic, km
v_int = v_kms[0]                   # km/s
print(f"integrator state @ MJD {TARGET_MJD} (heliocentric ecliptic):")
print(f"  r = {r_int} km   |r| = {np.linalg.norm(r_int)/AU_KM:.6f} AU")
print(f"  v = {v_int} km/s")

/astro/users/ds2004/.conda/envs/neofast_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


integrator state @ MJD 61642.0 (heliocentric ecliptic):
  r = [-2.29299073e+08 -5.09975432e+08 -1.43784987e+07] km   |r| = 3.738948 AU
  v = [10.9526308   1.2712449   0.65372597] km/s


## 3. Horizons truth at the target time, and position comparison

In [4]:
def horizons_state(obj, mjd_tdb):
    """Heliocentric ecliptic state (km, km/s) at mjd_tdb."""
    q = Horizons(id=obj, location='@sun', epochs=2400000.5 + mjd_tdb, id_type='smallbody')
    v = q.vectors(refplane='ecliptic')[0]
    r = np.array([float(v['x']), float(v['y']), float(v['z'])]) * AU_KM
    vv = np.array([float(v['vx']), float(v['vy']), float(v['vz'])]) * AU_KM / DAY_S
    return r, vv

r_hz, v_hz = horizons_state(OBJECT, TARGET_MJD)
dr = np.linalg.norm(r_int - r_hz)
dv = np.linalg.norm(v_int - v_hz)
delta_au = np.linalg.norm(r_hz) 
ang_arcsec = np.degrees(dr / np.linalg.norm(r_hz)) * 3600   # heliocentric angular, rough
print(f"Horizons state @ MJD {TARGET_MJD}:")
print(f"  r = {r_hz} km   |r| = {np.linalg.norm(r_hz)/AU_KM:.6f} AU")
print()
print(f"|dr| integrator - Horizons = {dr:.3f} km  ({dr/AU_KM:.3e} AU)")
print(f"|dv|                       = {dv*1000:.4f} m/s")
print(f"span = {(TARGET_MJD-EPOCH_MJD)/365.25:.2f} yr  ->  {dr/((TARGET_MJD-EPOCH_MJD)/365.25):.2f} km/yr")

Horizons state @ MJD 61642.0:
  r = [-2.29299073e+08 -5.09975432e+08 -1.43784987e+07] km   |r| = 3.738948 AU

|dr| integrator - Horizons = 0.140 km  (9.391e-10 AU)
|dv|                       = 0.0000 m/s
span = 2.85 yr  ->  0.05 km/yr


## 4. Recovered elements at the target time vs Horizons

In [5]:
def rv_to_elements(r, v, mu=GM_SUN):
    """Classical osculating elements from heliocentric state (km, km/s)."""
    R = np.linalg.norm(r); V = np.linalg.norm(v)
    h = np.cross(r, v); H = np.linalg.norm(h)
    n = np.cross([0,0,1.0], h); N = np.linalg.norm(n)
    evec = (np.cross(v, h)/mu) - r/R; e = np.linalg.norm(evec)
    a = 1.0 / (2.0/R - V*V/mu)
    i = np.degrees(np.arccos(h[2]/H))
    Om = np.degrees(np.arctan2(n[1], n[0])) % 360
    w  = np.degrees(np.arccos(np.clip(np.dot(n, evec)/(N*e), -1, 1)))
    if evec[2] < 0: w = 360 - w
    return dict(a=a/AU_KM, e=e, incl=i, Omega=Om % 360, w=w % 360)

el_int = rv_to_elements(r_int, v_int)
el_hz  = horizons_elements(OBJECT, TARGET_MJD)

import pandas as pd
rows = []
for k, unit in [('a','AU'), ('e',''), ('incl','deg'), ('Omega','deg'), ('w','deg')]:
    iv, hv = el_int[k], el_hz[k]
    rows.append({'element': f'{k} ({unit})'.strip(), 'integrator': round(iv,8),
                 'Horizons': round(hv,8), 'diff': f'{iv-hv:+.2e}'})
display(pd.DataFrame(rows).set_index('element'))

,integrator,Horizons,diff
element,,,
a (AU),2.516462,2.516462,+5.78e-11
e (),0.661078,0.661078,-9.20e-11
incl (deg),3.407233,3.407233,-8.82e-10
Omega (deg),271.378668,271.378668,+6.79e-09
w (deg),134.342595,134.342595,-9.05e-09


## 5. Drift sweep — how the error grows with propagation span

Same as the Apophis check in `fixing_integrator.md` §9.9C: propagate the fixed start-epoch
elements to a range of target times and report |dr| vs Horizons. Pre-encounter this should
stay ~km/yr; a deep planetary encounter amplifies it (chaos, not a bug).

In [6]:
offsets_yr = [0.25, 0.5, 1.0, 2.0, 3.0]
rows = []
for dy in offsets_yr:
    tgt = EPOCH_MJD + dy * 365.25
    rk, _ = vdp.propagate_elements_nbody(
        a_AU=[el0['a']], e=[el0['e']], inc_deg=[el0['incl']], raan_deg=[el0['Omega']],
        argp_deg=[el0['w']], tp_mjd=[el0['Tp_mjd']], epoch_mjd=[el0['epoch_mjd']],
        obstime_str=Time(tgt, format='mjd', scale='tdb'), show_progress=False, gr_model=GR_MODEL)
    rh, _ = horizons_state(OBJECT, tgt)
    d = np.linalg.norm(rk[0] - rh)
    rows.append({'span_yr': dy, 'target_MJD': round(tgt,1), '|dr|_km': round(d,3), 'km/yr': round(d/dy,3)})
display(pd.DataFrame(rows).set_index('span_yr'))

,target_MJD,|dr|_km,km/yr
span_yr,,,
0.25,60691.3,0.003,0.014
0.50,60782.6,0.013,0.026
1.00,60965.2,0.036,0.036
2.00,61330.5,0.086,0.043
3.00,61695.8,0.153,0.051


## 6. Full observable comparison — RA, DEC, rates, ecliptic lon/lat, rates, mag, distance

Replicates the exact production geometry chain from `build_visible_subset_dataframe`
(origin fix: barycentric Earth minus barycentric Sun = heliocentric observer; manual
equatorial<->ecliptic rotation that preserves differentials) so every quantity VDP
actually uses can be checked against Horizons in one table. Reports both **no light-time**
(what `gmm.py` uses today, §9.3 gap #2) and a **light-time-corrected** version (2-iteration
linear correction, matching Stage 0's approach), since §9.9 showed light-time is the dominant
residual term.

Horizons side queried at **location='X05'** (Rubin/Vera Rubin Observatory), matching
`rubin_location` in `neoscore.py` — topocentric, apples-to-apples.

Convention note: Horizons' `RA_rate` is `d(RA)/dt * cos(DEC)`; the production `dra_deg_day`
is the **raw** `d(RA)/dt` (no cosDec) — converted explicitly below (the same conversion
noted in `fixing_integrator.md` section 12.3: `dra = RARateCosDec/cos(dec)`).

In [7]:
import astropy.units as u
from astropy.coordinates import get_body_barycentric_posvel, solar_system_ephemeris
from velocity_density_pipeline_gmm import ECLIPTIC_OBLIQUITY_DEG
import neoscore as nsc

C_KM_S = 299792.458  # speed of light

# ---- our side: replicate build_visible_subset_dataframe's geometry chain ----
eps = np.deg2rad(ECLIPTIC_OBLIQUITY_DEG)
cE, sE = np.cos(eps), np.sin(eps)

def ecl_to_equ(r_ecl, v_ecl):
    r_eq = np.array([r_ecl[0], cE*r_ecl[1] - sE*r_ecl[2], sE*r_ecl[1] + cE*r_ecl[2]])
    v_eq = np.array([v_ecl[0], cE*v_ecl[1] - sE*v_ecl[2], sE*v_ecl[1] + cE*v_ecl[2]])
    return r_eq, v_eq

def equ_to_ecl(r_eq, v_eq):
    r_ecl = np.array([r_eq[0], cE*r_eq[1] + sE*r_eq[2], -sE*r_eq[1] + cE*r_eq[2]])
    v_ecl = np.array([v_eq[0], cE*v_eq[1] + sE*v_eq[2], -sE*v_eq[1] + cE*v_eq[2]])
    return r_ecl, v_ecl

def radec_rates_from_relvec(r_rel, v_rel):
    x, y, z = r_rel
    vx, vy, vz = v_rel
    rho2 = x*x + y*y
    r2 = rho2 + z*z
    rho = np.sqrt(rho2)
    ra = np.arctan2(y, x)
    dec = np.arctan2(z, rho)
    dra = (x*vy - y*vx) / rho2          # raw d(RA)/dt, no cosDec
    ddec = (vz*rho2 - z*(x*vx + y*vy)) / (r2*rho)
    return (np.rad2deg(ra) % 360, np.rad2deg(dec),
            np.rad2deg(dra)*86400.0, np.rad2deg(ddec)*86400.0)

def lamb_rates_from_relvec(r_rel_ecl, v_rel_ecl):
    x, y, z = r_rel_ecl
    vx, vy, vz = v_rel_ecl
    rho2 = x*x + y*y
    r2 = rho2 + z*z
    rho = np.sqrt(rho2)
    lam = np.arctan2(y, x)
    beta = np.arctan2(z, rho)
    dlam = (x*vy - y*vx) / rho2
    dbeta = (vz*rho2 - z*(x*vx + y*vy)) / (r2*rho)
    return (np.rad2deg(lam) % 360, np.rad2deg(beta),
            np.rad2deg(dlam)*86400.0, np.rad2deg(dbeta)*86400.0)

t_obs = Time(TARGET_MJD, format='mjd', scale='tdb')

# heliocentric ecliptic state of the object (n-body, already computed as r_int, v_int above)
r_obj_helio_ecl, v_obj_helio_ecl = r_int.copy(), v_int.copy()
r_obj_helio, v_obj_helio = ecl_to_equ(r_obj_helio_ecl, v_obj_helio_ecl)

# observer: barycentric Earth + Rubin topocentric offset, corrected to heliocentric origin (the section 9.2 fix)
# NOTE: for a scalar Time, _get_earth_and_observer already returns shape (3,) -- do NOT index with [0].
scorer = nsc.NEOMODScorer(None, None, None, None, None)
_, rE, vE, r_tele = scorer._get_earth_and_observer(t_obs)
with solar_system_ephemeris.set('de432s'):
    r_sun_bary, v_sun_bary = get_body_barycentric_posvel('sun', t_obs)
r_sun_bary = r_sun_bary.xyz.to_value(u.km)
v_sun_bary = v_sun_bary.xyz.to_value(u.km/u.s)
r_observer_helio = rE - r_sun_bary + r_tele
v_observer_helio = vE - v_sun_bary

# ---- no light-time (what gmm.py uses today) ----
r_rel_noltc = r_obj_helio - r_observer_helio
v_rel_noltc = v_obj_helio - v_observer_helio
ra_n, dec_n, dra_n, ddec_n = radec_rates_from_relvec(r_rel_noltc, v_rel_noltc)
r_rel_ecl_n, v_rel_ecl_n = equ_to_ecl(r_rel_noltc, v_rel_noltc)
lam_n, beta_n, vlam_n, vbeta_n = lamb_rates_from_relvec(r_rel_ecl_n, v_rel_ecl_n)
delta_n = np.linalg.norm(r_rel_noltc) / AU_KM

# ---- light-time corrected (2-iteration linear correction, matches Stage 0) ----
lt_s = np.linalg.norm(r_rel_noltc) / C_KM_S
for _ in range(2):
    r_obj_helio_lt = r_obj_helio - v_obj_helio * lt_s
    r_rel_ltc = r_obj_helio_lt - r_observer_helio
    lt_s = np.linalg.norm(r_rel_ltc) / C_KM_S
v_rel_ltc = v_obj_helio - v_observer_helio   # curvature over lt negligible (fixing_integrator.md sec 10.8)
ra_l, dec_l, dra_l, ddec_l = radec_rates_from_relvec(r_rel_ltc, v_rel_ltc)
r_rel_ecl_l, v_rel_ecl_l = equ_to_ecl(r_rel_ltc, v_rel_ltc)
lam_l, beta_l, vlam_l, vbeta_l = lamb_rates_from_relvec(r_rel_ecl_l, v_rel_ecl_l)
delta_l = np.linalg.norm(r_rel_ltc) / AU_KM

print(f"light_time = {lt_s:.3f} s")
print(f"no-LTC:  RA={ra_n:.6f} DEC={dec_n:.6f}  lam={lam_n:.6f} beta={beta_n:.6f}")
print(f"LTC:     RA={ra_l:.6f} DEC={dec_l:.6f}  lam={lam_l:.6f} beta={beta_l:.6f}")

light_time = 1893.527 s
no-LTC:  RA=227.533250 DEC=-19.242356  lam=230.383370 beta=-1.451724
LTC:     RA=227.531739 DEC=-19.242047  lam=230.381912 beta=-1.451810


In [8]:
def horizons_observables(obj, mjd_utc, location='X05'):
    q = Horizons(id=obj, location=location, epochs=2400000.5 + mjd_utc, id_type='smallbody')
    eph = q.ephemerides(quantities='1,3,9,20')[0]
    return dict(
        RA_deg=float(eph['RA']), DEC_deg=float(eph['DEC']),
        RARateCosDec_deg_day=float(eph['RA_rate']) * (u.arcsec/u.hour).to(u.deg/u.day),
        DECrate_deg_day=float(eph['DEC_rate']) * (u.arcsec/u.hour).to(u.deg/u.day),
        delta_au=float(eph['delta']), delta_rate_au_day=float(eph['delta_rate']) * (u.km/u.s).to(u.AU/u.day),
        V_mag=float(eph['V']) if 'V' in eph.colnames else np.nan,
    )

hz = horizons_observables(OBJECT, TARGET_MJD)
for k, v in hz.items(): print(f"  {k:24s} = {v}")

# raw dRA/dt (no cosDec), to match our convention
hz['dra_deg_day'] = hz['RARateCosDec_deg_day'] / np.cos(np.radians(hz['DEC_deg']))
hz['ddec_deg_day'] = hz['DECrate_deg_day']

# reconstruct Horizons' full equatorial cartesian relative state from (RA, DEC, delta, rates)
ra_r, dec_r = np.radians(hz['RA_deg']), np.radians(hz['DEC_deg'])
dra_r  = np.radians(hz['dra_deg_day']) / 86400.0     # rad/s
ddec_r = np.radians(hz['ddec_deg_day']) / 86400.0
delta_km = hz['delta_au'] * AU_KM
ddelta_kms = hz['delta_rate_au_day'] * AU_KM / 86400.0

x_h = delta_km*np.cos(dec_r)*np.cos(ra_r)
y_h = delta_km*np.cos(dec_r)*np.sin(ra_r)
z_h = delta_km*np.sin(dec_r)
vx_h = (ddelta_kms*np.cos(dec_r)*np.cos(ra_r) - delta_km*np.sin(dec_r)*ddec_r*np.cos(ra_r)
        - delta_km*np.cos(dec_r)*np.sin(ra_r)*dra_r)
vy_h = (ddelta_kms*np.cos(dec_r)*np.sin(ra_r) - delta_km*np.sin(dec_r)*ddec_r*np.sin(ra_r)
        + delta_km*np.cos(dec_r)*np.cos(ra_r)*dra_r)
vz_h = ddelta_kms*np.sin(dec_r) + delta_km*np.cos(dec_r)*ddec_r

r_rel_hz = np.array([x_h, y_h, z_h])
v_rel_hz = np.array([vx_h, vy_h, vz_h])
r_rel_ecl_hz, v_rel_ecl_hz = equ_to_ecl(r_rel_hz, v_rel_hz)
lam_hz, beta_hz, vlam_hz, vbeta_hz = lamb_rates_from_relvec(r_rel_ecl_hz, v_rel_ecl_hz)
print(f"\nreconstructed Horizons ecliptic: lam={lam_hz:.6f} beta={beta_hz:.6f} vlam={vlam_hz:.6f} vbeta={vbeta_hz:.6f}")

  RA_deg                   = 227.53183
  DEC_deg                  = -19.24206
  RARateCosDec_deg_day     = 0.1134498
  DECrate_deg_day          = -0.020814266666666668
  delta_au                 = 3.79461749231828
  delta_rate_au_day        = 0.012182766876774803
  V_mag                    = 30.537

reconstructed Horizons ecliptic: lam=230.381998 beta=-1.451799 vlam=0.114908 vbeta=0.010430


In [9]:
# ---- magnitude: same HG formula gmm.py uses, fed our own state (independent of Horizons' V) ----
q_hg = Horizons(id=OBJECT, location='X05', epochs=2400000.5+TARGET_MJD, id_type='smallbody')
eph_hg = q_hg.ephemerides(quantities='1,9')[0]
H_obj = float(eph_hg['H']); G_obj = float(eph_hg['G']) if np.isfinite(eph_hg['G']) else 0.15

def hg_mag(H, G, r_sun_au, delta_au, alpha_rad):
    phi1 = np.exp(-3.33*np.tan(0.5*alpha_rad)**0.63)
    phi2 = np.exp(-1.87*np.tan(0.5*alpha_rad)**1.22)
    Phi = max((1-G)*phi1 + G*phi2, 1e-30)
    return H + 5*np.log10(r_sun_au*delta_au) - 2.5*np.log10(Phi)

def phase_angle(r_obj_helio, r_rel):
    u_sun = -r_obj_helio/np.linalg.norm(r_obj_helio)
    u_obs = -r_rel/np.linalg.norm(r_rel)
    return np.arccos(np.clip(np.dot(u_sun,u_obs), -1, 1))

r_sun_au = np.linalg.norm(r_obj_helio)/AU_KM
alpha_n = phase_angle(r_obj_helio, r_rel_noltc)
alpha_l = phase_angle(r_obj_helio_lt, r_rel_ltc)
mag_n = hg_mag(H_obj, G_obj, r_sun_au, delta_n, alpha_n)
mag_l = hg_mag(H_obj, G_obj, np.linalg.norm(r_obj_helio_lt)/AU_KM, delta_l, alpha_l)
print(f"H={H_obj}  G={G_obj}")
print(f"our mag_app: no-LTC={mag_n:.4f}  LTC={mag_l:.4f}   Horizons V={hz['V_mag']:.4f}")

H=23.93  G=0.15
our mag_app: no-LTC=30.5370  LTC=30.5371   Horizons V=30.5370


In [10]:
# ---- the full comparison table ----
rows = [
    ('RA (deg)',                hz['RA_deg'],               ra_n,   ra_l),
    ('DEC (deg)',                hz['DEC_deg'],              dec_n,  dec_l),
    ('RARateCosDec (deg/day)',  hz['RARateCosDec_deg_day'], dra_n*np.cos(np.radians(dec_n)), dra_l*np.cos(np.radians(dec_l))),
    ('DEC_rate (deg/day)',      hz['DECrate_deg_day'],      ddec_n, ddec_l),
    ('dRA/dt raw (deg/day)',    hz['dra_deg_day'],          dra_n,  dra_l),
    ('ecl lon lam (deg)',       lam_hz,                     lam_n,  lam_l),
    ('ecl lat beta (deg)',      beta_hz,                    beta_n, beta_l),
    ('vlam = dlam/dt (deg/day)',vlam_hz,                    vlam_n, vlam_l),
    ('vbeta = dbeta/dt (deg/day)', vbeta_hz,                vbeta_n, vbeta_l),
    ('delta (AU)',               hz['delta_au'],             delta_n, delta_l),
    ('V mag',                    hz['V_mag'],                mag_n,  mag_l),
]
tbl = pd.DataFrame(rows, columns=['quantity', 'Horizons', 'ours_noLTC', 'ours_LTC'])
tbl['diff_noLTC'] = tbl.ours_noLTC - tbl.Horizons
tbl['diff_LTC']   = tbl.ours_LTC   - tbl.Horizons
pd.set_option('display.float_format', lambda v: f'{v:.6f}')
display(tbl.set_index('quantity'))

# angular-quantity residuals in arcsec, for the position/rate rows
print(f"\n{'quantity':28s} {'diff_noLTC':>14s} {'diff_LTC':>14s}   (arcsec, for deg-valued rows)")
for _, r in tbl.iterrows():
    if 'deg' in r.quantity or r.quantity in ('RA (deg)','DEC (deg)'):
        print(f"{r.quantity:28s} {r.diff_noLTC*3600:14.4f} {r.diff_LTC*3600:14.4f}")

,Horizons,ours_noLTC,ours_LTC,diff_noLTC,diff_LTC
quantity,,,,,
RA (deg),227.531830,227.533250,227.531739,0.001420,-0.000091
DEC (deg),-19.242060,-19.242356,-19.242047,-0.000296,0.000013
RARateCosDec (deg/day),0.113450,0.116292,0.116293,0.002842,0.002843
DEC_rate (deg/day),-0.020814,-0.021711,-0.021713,-0.000897,-0.000898
dRA/dt raw (deg/day),0.120163,0.123173,0.123174,0.003010,0.003011
ecl lon lam (deg),230.381998,230.383370,230.381912,0.001372,-0.000086
ecl lat beta (deg),-1.451799,-1.451724,-1.451810,0.000075,-0.000011
vlam = dlam/dt (deg/day),0.114908,0.117887,0.117888,0.002979,0.002981
vbeta = dbeta/dt (deg/day),0.010430,0.010329,0.010329,-0.000101,-0.000102



quantity                         diff_noLTC       diff_LTC   (arcsec, for deg-valued rows)
RA (deg)                             5.1133        -0.3285
DEC (deg)                           -1.0652         0.0451
RARateCosDec (deg/day)              10.2303        10.2348
DEC_rate (deg/day)                  -3.2291        -3.2342
dRA/dt raw (deg/day)                10.8364        10.8404
ecl lon lam (deg)                    4.9379        -0.3110
ecl lat beta (deg)                   0.2709        -0.0399
vlam = dlam/dt (deg/day)            10.7254        10.7309
vbeta = dbeta/dt (deg/day)          -0.3652        -0.3655
